In [1]:
import numpy as np
import random
import os
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score

# ─── 1. CHARGEMENT DU DICTIONNAIRE ────────────────────────
# Le dictionnaire doit contenir des mots écrits avec les noms de classes AMHCD
# (ex: "ya-yab-yach" ou des mots Tifinagh selon ton lexique disponible)
lexique_path = "lexique.txt"

if os.path.exists(lexique_path):
    with open(lexique_path, "r", encoding="utf-8") as f:
        dictionnaire = [line.strip() for line in f if line.strip()]
    print(f"Dictionnaire chargé : {len(dictionnaire)} mots uniques.")
else:
    # Backup : mots construits à partir des classes AMHCD
    dictionnaire = ["ya", "yab", "yach", "yad", "yae", "yaf", "yag",
                    "yah", "yaj", "yak", "yal", "yam", "yan", "yaq",
                    "yar", "yas", "yat", "yaw", "yax", "yay", "yaz"]
    print(f"Lexique de secours utilisé : {len(dictionnaire)} mots.")


C:\Users\fatima zehra\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\fatima zehra\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\fatima zehra\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_

Dictionnaire chargé : 1829 mots uniques.


In [2]:
# ─── 2. DISTANCE DE LEVENSHTEIN ───────────────────────────
def levenshtein(s1, s2):
    """Calcule la distance d'édition entre deux chaînes."""
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[m][n]

In [3]:
# ─── 3. CORRECTEUR AUTOMATIQUE ────────────────────────────
def corriger_mot(mot_errone, dictionnaire, top_k=3):
    """Trouve les mots les plus proches dans le dictionnaire."""
    distances = sorted(
        [(mot, levenshtein(mot_errone, mot)) for mot in dictionnaire],
        key=lambda x: x[1]
    )
    return distances[0][0], distances[:top_k]

In [4]:
# ─── 4. SIMULATION D'ERREURS OCR ────────────────────────── 
class_names = list(np.load("../data/class_names.npy"))

def simuler_erreur_ocr(mot, taux_erreur=0.2):
    """Simule une erreur OCR en remplaçant des caractères aléatoirement."""
    mot_liste = list(mot)
    nb_erreurs = max(1, int(len(mot) * taux_erreur))
    for _ in range(nb_erreurs):
        idx = random.randint(0, len(mot_liste) - 1)
        # Remplace par une lettre proche (a, e, i, o, u, b, d...)
        substitutions = [c for c in "abcdefghijklmnopqrstuvwxyz" if c != mot_liste[idx]]
        mot_liste[idx] = random.choice(substitutions)
    return ''.join(mot_liste)

In [5]:
# ─── 5. TEST DU MODULE DE CORRECTION ──────────────────────
print("\n" + "="*60)
print("TEST DU MODULE DE CORRECTION AUTOMATIQUE")
print("="*60)

random.seed(42)
mots_test = random.sample(dictionnaire, min(10, len(dictionnaire)))

corrections_reussies = 0
resultats = []

for mot_original in mots_test:
    mot_errone = simuler_erreur_ocr(mot_original, taux_erreur=0.25)
    mot_corrige, suggestions = corriger_mot(mot_errone, dictionnaire)

    succes = (mot_corrige == mot_original)
    if succes:
        corrections_reussies += 1

    resultats.append({
        'original': mot_original,
        'errone':   mot_errone,
        'corrige':  mot_corrige,
        'succes':   succes,
    })

    statut = "✅" if succes else "❌"
    print(f"\n{statut}  Original   : {mot_original}")
    print(f"    Erroné     : {mot_errone}")
    print(f"    Corrigé    : {mot_corrige}")
    print(f"    Suggestions: {[s[0] for s in suggestions]}")



TEST DU MODULE DE CORRECTION AUTOMATIQUE

✅  Original   : ⵜⴰⵎⵙⵓⵍⵜⴰ
    Erroné     : ⵜsⵎⵙⵓⵍbⴰ
    Corrigé    : ⵜⴰⵎⵙⵓⵍⵜⴰ
    Suggestions: ['ⵜⴰⵎⵙⵓⵍⵜⴰ', 'ⵜⵉⵎⵓⵔⴰ', 'ⵜⵎⵎⵓⵣⵖⴰ']

✅  Original   : ⴰⵢⵍⴰⵍ
    Erroné     : cⵢⵍⴰⵍ
    Corrigé    : ⴰⵢⵍⴰⵍ
    Suggestions: ['ⴰⵢⵍⴰⵍ', 'ⴰⵙⵍⴰⵍ', 'ⴰⵢⵍⴰⵏ']

✅  Original   : ⴰⴼⵕⴰⵏⵚⵉⵚ
    Erroné     : rⴼⵕhⵏⵚⵉⵚ
    Corrigé    : ⴰⴼⵕⴰⵏⵚⵉⵚ
    Suggestions: ['ⴰⴼⵕⴰⵏⵚⵉⵚ', 'ⴼⵕⴰⵏⵚⴰ', 'ⵜⴰⴼⵕⴰⵏⵚⵉⵚⵜ']

✅  Original   : ⵜⵉⵍⴰⵜⵉⵏⵉⵏ
    Erroné     : ⵜⵉⵍwⵜⵉⵏⵉn
    Corrigé    : ⵜⵉⵍⴰⵜⵉⵏⵉⵏ
    Suggestions: ['ⵜⵉⵍⴰⵜⵉⵏⵉⵏ', 'ⵜⴰⵍⴰⵜⵉⵏⵉⵜ', 'ⵜⵍⴰⵜⵉⵏⵉⵜ']

✅  Original   : ⵉⵎⵍⵢⵓⵏⴻⵏ
    Erroné     : ⵉⵎⵍozⵏⴻⵏ
    Corrigé    : ⵉⵎⵍⵢⵓⵏⴻⵏ
    Suggestions: ['ⵉⵎⵍⵢⵓⵏⴻⵏ', 'ⵉⵎⵍⵢⵓⵏⵏ', 'ⵉⵎⴳⵓⵏⵏ']

✅  Original   : ⵉⴷⵍⵉⵙⵏ
    Erroné     : yⴷⵍⵉⵙⵏ
    Corrigé    : ⵉⴷⵍⵉⵙⵏ
    Suggestions: ['ⵉⴷⵍⵉⵙⵏ', 'ⴰⴷⵍⵉⵙ', 'ⵉⴷⵍⵍⵉⵙⵏ']

✅  Original   : ⵅⴼⴰⵡⵍⵏ
    Erroné     : ⵅwⴰⵡⵍⵏ
    Corrigé    : ⵅⴼⴰⵡⵍⵏ
    Suggestions: ['ⵅⴼⴰⵡⵍⵏ', 'ⵙⴰⵡⵍⵏ', 'ⵉⵙⴰⵡⴰⵍⵏ']

✅  Original   : ⴱⵓⴷⵔⴰⵄ
    Erroné     : ⴱⵓⴷkⴰⵄ
    Corrigé    : ⴱⵓⴷⵔⴰⵄ
    S

In [6]:
# ─── 6. MÉTRIQUES DU MODULE NLP ───────────────────────────
taux_correction = corrections_reussies / len(mots_test) * 100

wer_avant = sum(1 for r in resultats if r['errone']  != r['original']) / len(resultats)
wer_apres = sum(1 for r in resultats if r['corrige'] != r['original']) / len(resultats)

print("\n" + "="*60)
print("MÉTRIQUES DU MODULE NLP")
print("="*60)
print(f"Mots testés          : {len(mots_test)}")
print(f"Corrections réussies : {corrections_reussies}")
print(f"Taux de correction   : {taux_correction:.1f}%")
print(f"WER avant correction : {wer_avant*100:.1f}%")
print(f"WER après correction : {wer_apres*100:.1f}%")
print(f"Amélioration WER     : {(wer_avant - wer_apres)*100:.1f}%")




MÉTRIQUES DU MODULE NLP
Mots testés          : 10
Corrections réussies : 10
Taux de correction   : 100.0%
WER avant correction : 100.0%
WER après correction : 0.0%
Amélioration WER     : 100.0%


In [7]:
# ─── 7. ÉVALUATION CNN SUR LE DATASET AMHCD ───────────────
print("\n" + "="*60)
print("ÉVALUATION CNN — DATASET AMHCD")
print("="*60)

model  = load_model("../results/models/best_model.h5")
X_test = np.load("../data/X_test.npy")
y_test = np.load("../data/y_test.npy")
y_true = np.argmax(y_test, axis=1)

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
acc    = accuracy_score(y_true, y_pred)

print(f"Test Accuracy (images réelles AMHCD) : {acc*100:.2f}%")



ÉVALUATION CNN — DATASET AMHCD


Test Accuracy (images réelles AMHCD) : 99.43%


In [8]:
# ─── 8. PIPELINE OCR → CORRECTION ────────────────────────
def pipeline_ocr_correction(images_sequence, model, dictionnaire, class_names):
    """
    Enchaîne : CNN (reconnaissance) → Levenshtein (correction).
    Entrée  : séquence d'images de lettres (N, 28, 28, 1)
    Sortie  : (mot_brut, mot_corrigé, correction_appliquée)
    """
    predictions   = model.predict(images_sequence, verbose=0)
    indices       = np.argmax(predictions, axis=1)
    mot_reconnu   = ''.join([class_names[i] for i in indices])

    if mot_reconnu in dictionnaire:
        return mot_reconnu, mot_reconnu, False
    else:
        mot_final, _ = corriger_mot(mot_reconnu, dictionnaire)
        return mot_reconnu, mot_final, True


In [9]:
# ─── 9. SYNTHÈSE FINALE ───────────────────────────────────
print("\n" + "="*60)
print("SYNTHÈSE DES PERFORMANCES DU SYSTÈME")
print("="*60)
print(f" 1. VISION (CNN) :")
print(f"    • Dataset           : AMHCD (images manuscrites réelles)")
print(f"    • Nb classes        : {len(class_names)}")
print(f"    • Test Accuracy     : {acc*100:.2f}%")
print(f"\n 2. NLP (LEVENSHTEIN) :")
print(f"    • Taille du lexique : {len(dictionnaire)} mots")
print(f"    • Taux correction   : {taux_correction:.1f}%")
print(f"\n 3. IMPACT GLOBAL :")
print(f"    • WER avant         : {wer_avant*100:.1f}%")
print(f"    • WER après         : {wer_apres*100:.1f}%")
if wer_avant > 0:
    print(f"    • Gain de fiabilité : {((wer_avant - wer_apres)/wer_avant)*100:.1f}%")
print("\n Pipeline OCR → Correction opérationnel !")


SYNTHÈSE DES PERFORMANCES DU SYSTÈME
 1. VISION (CNN) :
    • Dataset           : AMHCD (images manuscrites réelles)
    • Nb classes        : 33
    • Test Accuracy     : 99.43%

 2. NLP (LEVENSHTEIN) :
    • Taille du lexique : 1829 mots
    • Taux correction   : 100.0%

 3. IMPACT GLOBAL :
    • WER avant         : 100.0%
    • WER après         : 0.0%
    • Gain de fiabilité : 100.0%

 Pipeline OCR → Correction opérationnel !
